### Using Spark SQL 

In [0]:
%python
spark

In [0]:
%sql
select prd_id,
 replace(substr(prd_key,1,5),'-','_') as cat_id,
 substr(prd_key,7,len(prd_key)) as prd_key,
 ifnull(prd_cost,0) as prd_cost,
 case upper(TRIM(prd_line)) when 'M' then "Mountain"
    when 'R' then "Road"
    when 'S' then "Other Sales"
    when 'T' then "Touring"
    else 'NA' end as prd_line,
prd_start_dt,
lead(prd_start_dt) over (partition by prd_key order by prd_start_dt)-1 as prd_end_dt
 from bronze.crm_prd_info
 order by prd_key

### Using Spark Select and Expr

In [0]:
%python
from pyspark.sql.functions import expr,substr,replace,col,lead,substring,lit,length,when,trim
from pyspark.sql.window import Window

window_spec = Window.partitionBy("prd_key").orderBy("prd_start_dt")


df = spark.table("bronze.crm_prd_info")
df.printSchema()

df_final = df.withColumn("prd_end_dt",(lead(col("prd_start_dt")).over(window_spec)-1))\
                .withColumn("cat_id",replace(substring(col("prd_key"),1,5),lit('-'),lit('_')))\
                .withColumn("prd_key",substring(col("prd_key"),7,length(col("prd_key"))))\
                .withColumn("prd_cost",when(col("prd_cost").isNull(),0).otherwise(col("prd_cost")))\
                .withColumn("prd_line",when(trim(col("prd_line"))=='M','Mountain').when(trim(col("prd_line"))=='R','Road').when(trim(col("prd_line"))=='S','Other Sales').when(trim(col("prd_line"))=='T','Touring').otherwise(lit("NA")))\
                
display(df_final)


In [0]:
%python
df_final = df_final.select("prd_id","cat_id","prd_key","prd_nm","prd_cost","prd_line","prd_start_dt","prd_end_dt")

In [0]:
%python
df_final.write.format('delta').mode("overwrite").save("dbfs:/FileStore/tables/warehouse/source_crm/silverprd")

In [0]:
%sql

CREATE TABLE IF NOT EXISTS silver.crm_prd_info
USING DELTA
LOCATION "dbfs:/FileStore/tables/warehouse/source_crm/silverprd";

In [0]:
%sql
select * from silver.crm_prd_info